# CoCoOp Playground: image-conditional prompts (beginner annotated)

Paper: Zhou, Yang, Loy, Liu. *Conditional Prompt Learning for Vision-Language Models* (CVPR 2022).
Official code: https://github.com/KaiyangZhou/CoOp (CoCoOp lives in the same repo)

Every code line has a `# comment`. Read the markdown above each cell, then run it with Shift+Enter.
It's easier to follow if you have run the **CoOp notebook first**, since this one builds on it.

**Runtime:** `Runtime > Change runtime type > T4 GPU`. Total run time is roughly 10 to 15 minutes.

## 0. The idea in 2 minutes

**Recap of CoOp:** replace the hand-written prompt words "a photo of a" with M learnable vectors
`[V]1 ... [V]M`, keep CLIP frozen, train only those vectors on a few labelled images. It works well on the
training classes but **overfits**: on classes never seen during training it often gets *worse* than the
plain hand-written prompt. The learned context is *static*: the same for every image.

**CoCoOp (Conditional CoOp)** makes the context depend on the image. A tiny network called the
**Meta-Net** looks at each image's CLIP embedding and outputs a small shift vector `pi(x)` that is
added to every context vector:

```
                 image x
                    |
            CLIP image encoder (frozen)
                    |
              image embedding ------------------------------+
                    |                                        |
     Meta-Net: Linear(512->32) -> ReLU -> Linear(32->512)    |
                    |                                        |
                  pi(x)  (one 512-number shift)              |
                    |                                        |
  [SOS] [V1+pi] [V2+pi] [V3+pi] [V4+pi] [class] [.] [EOS]    |
                    |                                        |
           CLIP text encoder (frozen)                        |
                    |                                        |
          class text embedding  <---- cosine similarity -----+
```

Why this helps: instead of memorising "what words describe *these* classes", the context adapts to
each image, which transfers better to unseen classes.

**Trainable numbers:** 4 x 512 context vectors + about 33,000 Meta-Net weights. Still tiny.

**The price:** in CoOp, the class text embeddings are the same for all images, so the text encoder runs
once per step. In CoCoOp each image gets its own prompts, so the text encoder runs
**(number of images x number of classes)** times. That's why training uses small batches and takes longer.

In the paper (ViT-B/16, average of 11 datasets, base-to-new): CoOp went from about 82.7% on base classes to
63.2% on new ones (zero-shot CLIP: 74.2% on new), while CoCoOp got about 80.5% base and 71.7% new, raising H
from 71.7 to 75.8. Your CIFAR-100 numbers will differ, but look for the same pattern.

## 1. Setup

We use OpenAI's original `clip` package (not Hugging Face) because it lets us reach inside the text
encoder and feed it our own token vectors, which is exactly what prompt learning needs.
The official CoOp/CoCoOp code does the same.

In [ ]:
!pip -q install ftfy regex
!pip -q install git+https://github.com/openai/CLIP.git

In [ ]:
import os, math, time, random                 # standard library helpers
from glob import glob                          # find files by pattern, used for your own dataset
from collections import OrderedDict           # ordered dictionary, used to name Meta-Net layers

import numpy as np                             # arrays and math on the CPU
import torch                                   # PyTorch, the deep learning library
import torch.nn as nn                          # building blocks for neural networks (layers, Parameter)
import torch.nn.functional as F                # functions like cross_entropy and normalize
import matplotlib.pyplot as plt                # plotting
import requests                                # download images from URLs
from io import BytesIO                         # treat downloaded bytes like a file
from PIL import Image                          # open images
import clip                                    # OpenAI CLIP: model, preprocessing and tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"   # use the GPU if there is one
print("Device:", device)

def set_seed(seed):
    """Fix all random number generators so results are repeatable."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

## 2. Settings

All the knobs in one place. Keep the defaults for a first run. If you change `SEED`, `SHOTS`, `N_EVAL`,
`BACKBONE` or the dataset, change them in **both** notebooks so the comparison stays fair.

In [ ]:
SEED = 1                     # random seed; same value in both notebooks = same data split
BACKBONE = "ViT-B/32"        # CLIP model: "ViT-B/32" (fast), "ViT-B/16" (paper default, slower), "RN50"

DATASET = "cifar100"         # "cifar100" or "folder" (your own data, see section 3)
DATA_DIR = "/content/my_dataset"   # only used when DATASET = "folder"

SHOTS = 16                   # labelled training images per base class (few-shot)
N_EVAL = 2000                # max test images per split (base / new), to keep evaluation quick

N_CTX = 4                    # number of learnable context vectors (only used if CTX_INIT is empty)
CTX_INIT = "a photo of a"    # start the context from these words; set "" to start from random numbers

EPOCHS = 10           # passes over the training set (paper: 10)
BATCH_SIZE = 4        # images per step; small because the text encoder runs (images x classes) times
LR = 0.002            # learning rate: size of each update step (paper: 0.002)
EVAL_BATCH = 32       # images per step during evaluation (no gradients, so we can use more)

set_seed(SEED)

## 3. Load CLIP and freeze it

`clip.load` downloads the pretrained weights. On a GPU they come in half precision (16-bit floats);
`.float()` converts them to 32-bit, which is simpler and more stable for training.

**Freezing:** `requires_grad_(False)` tells PyTorch never to compute gradients for (and so never change)
CLIP's own weights. Only the parameters we create later will be trained.

In [ ]:
clip_model, preprocess = clip.load(BACKBONE, device=device, jit=False)   # model + image preprocessing function
clip_model = clip_model.float().eval()        # 32-bit weights; eval mode (no dropout etc.)
for p in clip_model.parameters():             # loop over every weight tensor in CLIP
    p.requires_grad_(False)                   # freeze it

# logit_scale is CLIP's learned "temperature" (about 100). Multiplying similarities by it
# makes the softmax probabilities sharper. .exp() because CLIP stores it as a log value.
logit_scale = clip_model.logit_scale.exp().detach()
print("Loaded", BACKBONE, "| text width:", clip_model.ln_final.weight.shape[0],
      "| image embedding dim:", clip_model.visual.output_dim)

## 4. Data: few-shot base classes, held-out new classes

**CIFAR-100** has 100 classes (apple, aquarium fish, baby, bear, ...), 500 training and 100 test images each.

What this cell builds:
- `classnames`: all class names, split into **base** (first half) and **new** (second half)
- a few-shot **training set**: `SHOTS` random training images from each *base* class
- two **test sets**: test images of base classes, and test images of new classes

**Your own data (`DATASET = "folder"`):** one sub-folder per class, like
`my_dataset/hole_in_fabric/*.jpg`. Each class's images are split 50/50 into a training pool and a test
pool. You need more than `SHOTS` images per class (the code lowers `SHOTS` automatically if needed).

In [ ]:
if DATASET == "cifar100":
    from torchvision.datasets import CIFAR100
    train_ds = CIFAR100("/content/data", train=True, download=True)     # 50,000 images, downloads once
    test_ds = CIFAR100("/content/data", train=False, download=True)     # 10,000 images
    classnames = [c.replace("_", " ") for c in train_ds.classes]         # "aquarium_fish" -> "aquarium fish"
    train_labels = np.array(train_ds.targets)                            # class number of every training image
    test_labels_all = np.array(test_ds.targets)
    # "getter" functions: given an index, return a PIL image. CIFAR stores pixels in a NumPy array.
    get_train = lambda i: Image.fromarray(train_ds.data[i])
    get_test = lambda i: Image.fromarray(test_ds.data[i])

elif DATASET == "folder":
    EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")
    class_dirs = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
    classnames = [d.replace("_", " ") for d in class_dirs]
    rng_split = np.random.default_rng(SEED)
    tr_paths, tr_y, te_paths, te_y = [], [], [], []
    for ci, d in enumerate(class_dirs):                                  # ci = class number
        ps = sorted(p for p in glob(os.path.join(DATA_DIR, d, "*")) if p.lower().endswith(EXTS))
        ps = list(rng_split.permutation(ps))                             # shuffle this class's images
        half = len(ps) // 2                                              # first half -> train pool
        tr_paths += ps[:half]; tr_y += [ci] * half
        te_paths += ps[half:]; te_y += [ci] * (len(ps) - half)
    train_labels, test_labels_all = np.array(tr_y), np.array(te_y)
    get_train = lambda i: Image.open(tr_paths[i]).convert("RGB")
    get_test = lambda i: Image.open(te_paths[i]).convert("RGB")
    SHOTS = min(SHOTS, int(np.bincount(train_labels).min()))            # can't take more shots than we have
    print("Using SHOTS =", SHOTS)
else:
    raise ValueError("DATASET must be 'cifar100' or 'folder'")

# ---- base / new split by class --------------------------------------------------------
n_cls = len(classnames)
n_base = math.ceil(n_cls / 2)                       # first half of the classes are "base"
base_names = classnames[:n_base]                    # seen during training
new_names = classnames[n_base:]                     # never seen during training
rng = np.random.default_rng(SEED)

# ---- few-shot training set: SHOTS images from each base class --------------------------
train_idx = np.concatenate([
    rng.choice(np.where(train_labels == c)[0], SHOTS, replace=False)   # SHOTS random images of class c
    for c in range(n_base)
])
y_train = train_labels[train_idx]                   # labels are 0..n_base-1 already

# ---- test sets ----------------------------------------------------------------------
def pick_test(class_ids):
    """Indices of test images whose class is in class_ids (at most N_EVAL of them)."""
    idx = np.where(np.isin(test_labels_all, class_ids))[0]
    return rng.choice(idx, N_EVAL, replace=False) if len(idx) > N_EVAL else idx

base_test_idx = pick_test(list(range(n_base)))
new_test_idx = pick_test(list(range(n_base, n_cls)))
y_base = test_labels_all[base_test_idx]             # 0..n_base-1
y_new = test_labels_all[new_test_idx] - n_base      # shift so new labels are also 0, 1, 2, ...

print(f"{n_cls} classes -> {len(base_names)} base, {len(new_names)} new")
print(f"train: {len(train_idx)} images ({SHOTS} per base class) | "
      f"test base: {len(base_test_idx)} | test new: {len(new_test_idx)}")
print("base examples:", base_names[:5], "| new examples:", new_names[:5])

### Encode all images once

The image encoder is frozen and we don't use data augmentation, so each image always gives the same
embedding. We compute every embedding **once** now and reuse it during training: this makes
training much faster, because only the text side has to run each step.

(The official code uses random crops and flips, which would require re-encoding each step.
Skipping them is a simplification that costs a little accuracy.)

In [ ]:
@torch.no_grad()                                    # no gradients needed: CLIP is frozen
def encode_images(getter, indices, bs=256):
    feats = []
    for i in range(0, len(indices), bs):            # go through the images in batches of 256
        # preprocess: resize to 224x224, crop, normalise pixel values -> tensor [3, 224, 224]
        # torch.stack puts the batch together -> [bs, 3, 224, 224]
        batch = torch.stack([preprocess(getter(j)) for j in indices[i:i + bs]]).to(device)
        f = clip_model.encode_image(batch)          # [bs, 512] image embeddings
        feats.append(F.normalize(f, dim=-1))        # scale each to length 1 (for cosine similarity)
    return torch.cat(feats)                         # [N, 512]

t0 = time.time()
f_train = encode_images(get_train, train_idx)       # few-shot training images
f_base = encode_images(get_test, base_test_idx)     # base-class test images
f_new = encode_images(get_test, new_test_idx)       # new-class test images
# labels as GPU tensors, so we can compare them with predictions directly
y_train_t = torch.tensor(y_train, device=device)
y_base_t = torch.tensor(y_base, device=device)
y_new_t = torch.tensor(y_new, device=device)
print(f"Encoded {len(f_train) + len(f_base) + len(f_new)} images in {time.time() - t0:.1f}s")

## 5. A text encoder that accepts our own token vectors

Normally you give CLIP a sentence, it **tokenizes** it (splits into word pieces, each mapped to an ID),
looks up each ID's vector in the **token embedding** table, and runs those vectors through a
Transformer. For prompt learning we need to **skip the lookup** and feed in vectors we made ourselves.
So we rebuild the text encoder's forward pass from CLIP's parts (as the official code does):

1. add **positional embeddings** (tell the Transformer each token's position)
2. run the **Transformer** (12 layers of attention)
3. apply the final **LayerNorm**
4. take the output at the **end-of-text (EOT) token** (the sentence summary) and **project** it to 512 numbers

**Speed trick:** CLIP pads every sentence to 77 tokens, but our prompts are about 10 tokens long. CLIP's
text Transformer is *causal* (each token only looks at tokens before it), so the padding after EOT can't
affect the EOT output. We cut the sequences to the longest real prompt in the batch. Same result,
about 5x faster.

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, cm):
        super().__init__()                                       # required first line for nn.Module subclasses
        self.transformer = cm.transformer                        # CLIP's 12-layer text Transformer
        self.positional_embedding = cm.positional_embedding      # [77, width] one vector per position
        self.ln_final = cm.ln_final                              # final layer normalisation
        self.text_projection = cm.text_projection                # [width, 512] projection matrix
        self.full_mask = cm.build_attention_mask().to(device)    # [77, 77] causal mask (-inf above diagonal)

    def forward(self, prompts, tokenized):
        # prompts:   [N, 77, width] token vectors (some of them learnable)
        # tokenized: [N, 77] token IDs, only used to find where EOT is
        eot = tokenized.argmax(dim=-1)            # EOT has the largest ID, so argmax = its position
        L = int(eot.max()) + 1                    # length of the longest prompt (speed trick)
        x = prompts[:, :L] + self.positional_embedding[:L]   # cut + add position information
        x = x.permute(1, 0, 2)                    # CLIP's Transformer wants [length, N, width]
        blocks = self.transformer.resblocks       # the 12 Transformer layers
        old = [b.attn_mask for b in blocks]       # remember the original 77x77 masks
        for b in blocks:
            b.attn_mask = self.full_mask[:L, :L]  # use a matching L x L causal mask
        try:
            x = self.transformer(x)               # run the frozen Transformer
        finally:                                  # always restore the masks, even if an error happens
            for b, m in zip(blocks, old):
                b.attn_mask = m
        x = x.permute(1, 0, 2)                    # back to [N, length, width]
        x = self.ln_final(x)
        # pick row i, position eot[i] for every prompt -> [N, width], then project -> [N, 512]
        return x[torch.arange(x.shape[0]), eot] @ self.text_projection

text_encoder = TextEncoder(clip_model)

@torch.no_grad()
def handcrafted_text_features(names, template="a photo of a {}."):
    """Zero-shot CLIP: embed hand-written prompts. Used as the baseline."""
    tok = clip.tokenize([template.format(n) for n in names]).to(device)   # [C, 77] token IDs
    emb = clip_model.token_embedding(tok)                                 # [C, 77, width] lookup
    return F.normalize(text_encoder(emb, tok), dim=-1)                    # [C, 512], length 1

def accuracy(logits, y):
    """Fraction of rows where the highest score is the correct class."""
    return (logits.argmax(-1) == y).float().mean().item()

def hmean(a, b):
    """Harmonic mean: high only if both a and b are high."""
    return 2 * a * b / (a + b) if a + b > 0 else 0.0

## 6. Baseline: zero-shot CLIP with the hand-written prompt

Before training anything, measure plain CLIP with `"a photo of a {class}."`.
Note: base images are classified among base classes only, new images among new classes only
(this is the standard base-to-new protocol).

In [ ]:
with torch.no_grad():
    zs_base = accuracy(logit_scale * f_base @ handcrafted_text_features(base_names).T, y_base_t)
    zs_new = accuracy(logit_scale * f_new @ handcrafted_text_features(new_names).T, y_new_t)

results = {"Zero-shot CLIP": (zs_base, zs_new, hmean(zs_base, zs_new))}   # we'll add more rows later
print(f"Zero-shot CLIP | base {zs_base*100:.2f}% | new {zs_new*100:.2f}% | H {hmean(zs_base, zs_new)*100:.2f}")

## 7. The prompt learner: context vectors + Meta-Net

Same as CoOp (`self.ctx`, 4 learnable vectors) plus the **Meta-Net**, a 2-layer network:
`Linear(512 -> 32) -> ReLU -> Linear(32 -> width)`. `ReLU` sets negative numbers to zero, which lets the
network learn non-linear functions. The narrow middle layer (512/16 = 32) keeps it small.

For a batch of B images and C classes, `forward` builds **B x C prompts**:

```
for every image b:   shift = MetaNet(image_feature[b])          # [width]
                     ctx_b = ctx + shift                        # [4, width], image-specific context
for every class c:   prompt[b, c] = [SOS] ctx_b  classname_c . [EOS]
```

**Broadcasting** (used below): `unsqueeze` adds a size-1 dimension, and `expand` repeats along it
without copying memory. That's how we pair every image's context with every class name.

In [ ]:
class PromptLearner(nn.Module):
    def __init__(self, n_ctx=4, ctx_init="a photo of a"):
        super().__init__()
        ctx_dim = clip_model.ln_final.weight.shape[0]      # width of token vectors (512 for ViT-B)
        if ctx_init:
            # Start from the real word vectors of ctx_init, so training begins at zero-shot quality
            tok = clip.tokenize(ctx_init).to(device)       # [1, 77]: [SOS] a photo of a [EOS] 0 0 ...
            n_ctx = int(tok.argmax(-1)) - 1                # number of tokens between SOS and EOS (4 here)
            with torch.no_grad():
                emb = clip_model.token_embedding(tok)      # [1, 77, width]
            ctx_vectors = emb[0, 1:1 + n_ctx].clone()      # the 4 word vectors -> [n_ctx, width]
            self.prefix_text = ctx_init                    # placeholder text used when tokenizing
        else:
            # Random start: small numbers from a normal distribution (std 0.02), like the paper
            ctx_vectors = torch.randn(n_ctx, ctx_dim, device=device) * 0.02
            self.prefix_text = " ".join(["X"] * n_ctx)     # "X X X X" just reserves n_ctx token slots
        self.n_ctx = n_ctx
        self.ctx = nn.Parameter(ctx_vectors)               # <- TRAINABLE context vectors
        # ---- Meta-Net: image embedding (vis_dim) -> one shift vector (ctx_dim) ----
        vis_dim = clip_model.visual.output_dim             # 512 for ViT-B models
        self.meta_net = nn.Sequential(OrderedDict([
            ("linear1", nn.Linear(vis_dim, vis_dim // 16)),     # 512 -> 32 numbers
            ("relu", nn.ReLU(inplace=True)),                    # keep positives, zero out negatives
            ("linear2", nn.Linear(vis_dim // 16, ctx_dim)),     # 32 -> width numbers
        ])).to(device)
        self._cache = {}                                   # class list -> fixed prompt parts

    @torch.no_grad()
    def _fixed_parts(self, classnames):
        """Token vectors before and after the context, for each class (computed once, then cached)."""
        key = tuple(classnames)
        if key not in self._cache:
            texts = [f"{self.prefix_text} {n}." for n in classnames]     # e.g. "a photo of a cat."
            tok = clip.tokenize(texts).to(device)                         # [C, 77] token IDs
            emb = clip_model.token_embedding(tok)                         # [C, 77, width]
            prefix = emb[:, :1]                                           # [SOS] vector, [C, 1, width]
            suffix = emb[:, 1 + self.n_ctx:]                              # class name, ".", [EOS], padding
            self._cache[key] = (prefix, suffix, tok)
        return self._cache[key]

    def forward(self, im_feats, classnames):
        """im_feats [B, 512] -> prompt vectors [B, C, 77, width] and token IDs [C, 77]."""
        prefix, suffix, tok = self._fixed_parts(classnames)     # [C,1,w], [C,77-1-n_ctx,w], [C,77]
        B, C = im_feats.shape[0], len(classnames)
        bias = self.meta_net(im_feats)                         # [B, width]: one shift per image
        ctx = self.ctx.unsqueeze(0) + bias.unsqueeze(1)        # [1,n_ctx,w] + [B,1,w] -> [B, n_ctx, w]
        ctx = ctx.unsqueeze(1).expand(-1, C, -1, -1)           # [B, C, n_ctx, w]: same per class
        prefix = prefix.unsqueeze(0).expand(B, -1, -1, -1)     # [B, C, 1, w]: same per image
        suffix = suffix.unsqueeze(0).expand(B, -1, -1, -1)     # [B, C, rest, w]
        prompts = torch.cat([prefix, ctx, suffix], dim=2)      # glue along the token axis -> [B, C, 77, w]
        return prompts, tok

set_seed(SEED)
prompt_learner = PromptLearner(N_CTX, CTX_INIT)
n_train = sum(p.numel() for p in prompt_learner.parameters())   # count trainable numbers
print(f"Context length: {prompt_learner.n_ctx} | trainable parameters: {n_train:,}")

## 8. From prompts to class scores

For B images and C classes we get B x C prompts. We flatten them into one big batch of B*C sentences,
run the text encoder once, then reshape back to `[B, C, 512]`. Each image is then compared only with
**its own** C class vectors. `torch.einsum("bd,bcd->bc", ...)` is a compact way to write
"for each image b and class c, dot product of image vector b with text vector (b, c)".

In [ ]:
def cocoop_logits(im_feats, names):
    prompts, tok = prompt_learner(im_feats, names)     # [B, C, 77, w], [C, 77]
    B, C, L, W = prompts.shape
    # flatten to [B*C, 77, w]; repeat the token IDs B times so they line up -> [B*C, 77]
    text_f = text_encoder(prompts.reshape(B * C, L, W), tok.repeat(B, 1))
    text_f = F.normalize(text_f, dim=-1).reshape(B, C, -1)   # back to [B, C, 512]
    return logit_scale * torch.einsum("bd,bcd->bc", im_feats, text_f)   # [B, C] scores

@torch.no_grad()
def eval_split(feats, y, names):
    """Evaluate in chunks, since every image needs its own C prompts."""
    correct = 0
    for i in range(0, len(feats), EVAL_BATCH):
        logits = cocoop_logits(feats[i:i + EVAL_BATCH], names)
        correct += (logits.argmax(-1) == y[i:i + EVAL_BATCH]).sum().item()
    return correct / len(feats)

def evaluate():
    b = eval_split(f_base, y_base_t, base_names)
    n = eval_split(f_new, y_new_t, new_names)          # unseen classes: Meta-Net adapts the context per image
    return b, n, hmean(b, n)

t0 = time.time()
b, n, h = evaluate()
print(f"Before training: base {b*100:.2f}% | new {n*100:.2f}% | H {h*100:.2f}  ({time.time()-t0:.0f}s)")
print("(Can be below zero-shot: the untrained Meta-Net already adds a random shift. Training fixes this.)")

## 9. Train

The standard PyTorch training loop:

1. **Forward:** compute logits for a batch of training images
2. **Loss:** `cross_entropy` measures how wrong the predictions are (low = correct class has high probability)
3. **Backward:** `loss.backward()` computes the **gradient** (which direction to change each trainable number)
4. **Step:** the optimizer (SGD with momentum) moves the trainable numbers a little in that direction

Extras:
- `weight_decay` gently pulls the numbers toward zero, which reduces overfitting
- the **cosine learning-rate schedule** starts at `LR` and smoothly decreases to 0, so updates get finer near the end
- `torch.randperm` shuffles the training images every epoch

Expect roughly 20 to 60 seconds per epoch on a T4.

In [ ]:
set_seed(SEED)
optimizer = torch.optim.SGD(prompt_learner.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
steps_per_epoch = math.ceil(len(f_train) / BATCH_SIZE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * steps_per_epoch)

history = []                                        # (loss, train accuracy) per epoch, for plotting
prompt_learner.train()                              # training mode
t0 = time.time()
for epoch in range(EPOCHS):
    perm = torch.randperm(len(f_train), device=device)   # random order of training images
    tot_loss, tot_correct = 0.0, 0
    for i in range(0, len(perm), BATCH_SIZE):
        b = perm[i:i + BATCH_SIZE]                  # indices of this batch
        logits = cocoop_logits(f_train[b], base_names)                              # 1. forward
        loss = F.cross_entropy(logits, y_train_t[b])        # 2. loss
        optimizer.zero_grad()                       # clear old gradients (PyTorch adds them up otherwise)
        loss.backward()                             # 3. backward: compute gradients
        optimizer.step()                            # 4. update the trainable numbers
        scheduler.step()                            # lower the learning rate a little
        tot_loss += loss.item() * len(b)            # .item() turns a 1-number tensor into a Python float
        tot_correct += (logits.argmax(-1) == y_train_t[b]).sum().item()
    history.append((tot_loss / len(f_train), tot_correct / len(f_train)))
    if epoch == 0 or (epoch + 1) % 1 == 0:   # print progress now and then
        print(f"epoch {epoch+1:>3}/{EPOCHS} | loss {history[-1][0]:.3f} | "
              f"train acc {history[-1][1]*100:.1f}% | {time.time()-t0:.0f}s")
prompt_learner.eval()                               # back to evaluation mode

In [ ]:
# Training curves: loss should go down, training accuracy up
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot([h[0] for h in history]); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
ax[1].plot([h[1] * 100 for h in history]); ax[1].set_title("training accuracy (%)"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 10. Evaluate: base vs new

Watch three numbers compared with zero-shot:
- **base**: should go up clearly (we trained on these classes)
- **new**: should stay closer to zero-shot than CoOp's did
- **H**: the harmonic mean, the headline metric for base-to-new generalisation

In [ ]:
b, n, h = evaluate()
results["CoCoOp"] = (b, n, h)
for k, (b_, n_, h_) in results.items():
    print(f"{k:<16} | base {b_*100:6.2f}% | new {n_*100:6.2f}% | H {h_*100:6.2f}")

### Compare with CoOp

Paste the CoOp numbers printed in section 10 of the CoOp notebook (as fractions, e.g. 0.80), with the same
settings, to put all three methods on one chart. Leave it as `None` to skip.

In [ ]:
COOP_RESULT = None          # e.g. (0.80, 0.55) = (base, new) from the CoOp notebook

if COOP_RESULT is not None:
    cb, cn = COOP_RESULT
    results["CoOp"] = (cb, cn, hmean(cb, cn))
    results = {k: results[k] for k in ["Zero-shot CLIP", "CoOp", "CoCoOp"]}   # fixed order for the chart

In [ ]:
# Grouped bar chart: one group per metric, one bar per method
labels = ["base", "new", "H"]
x = np.arange(len(labels))                         # positions 0, 1, 2
w = 0.8 / len(results)                             # bar width so the group fits in 0.8 units
fig, ax = plt.subplots(figsize=(7, 4))
for i, (name, vals) in enumerate(results.items()):
    bars = ax.bar(x + i * w - 0.4 + w / 2, [v * 100 for v in vals], w, label=name)
    ax.bar_label(bars, fmt="%.1f", fontsize=8)     # write the value on each bar
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("accuracy (%)")
ax.set_title(f"Base-to-new on {DATASET} ({BACKBONE}, {SHOTS}-shot)")
ax.legend(); plt.tight_layout(); plt.show()

## 11. What did it learn? Nearest words to the context vectors

The CoOp paper interprets learned vectors by finding the **closest real word vectors** in CLIP's
vocabulary (49,408 tokens) by Euclidean distance. They usually look like nonsense, which shows the best
prompt isn't necessarily something a human would write.

In [ ]:
from clip.simple_tokenizer import SimpleTokenizer
tokenizer = SimpleTokenizer()                      # converts token IDs back to text

with torch.no_grad():
    vocab = clip_model.token_embedding.weight     # [49408, width]: vector of every token in the vocabulary
    dist = torch.cdist(prompt_learner.ctx, vocab) # [n_ctx, 49408] distance from each context vector to each word
    nearest = dist.topk(5, largest=False)         # 5 smallest distances per context vector

for m in range(prompt_learner.n_ctx):
    words = [tokenizer.decoder[i].replace("</w>", "") for i in nearest.indices[m].tolist()]
    print(f"[V]{m+1}: " + ", ".join(f"{w} ({d:.2f})" for w, d in zip(words, nearest.values[m].tolist())))

### How much does the Meta-Net change the context?

Compare the size (length) of the image-specific shift with the size of the context vectors themselves.
The shift differs per image; this is what makes CoCoOp's prompts *conditional*.

In [ ]:
with torch.no_grad():
    shifts = prompt_learner.meta_net(f_base[:500])           # [500, width]: shift for 500 test images
    ctx_norm = prompt_learner.ctx.norm(dim=-1).mean().item() # average length of a context vector
    shift_norm = shifts.norm(dim=-1)                         # length of each image's shift
print(f"avg context vector length: {ctx_norm:.3f}")
print(f"avg shift length: {shift_norm.mean():.3f} (min {shift_norm.min():.3f}, max {shift_norm.max():.3f})")

plt.figure(figsize=(6, 3))
plt.hist(shift_norm.cpu().numpy(), bins=40)
plt.xlabel("length of Meta-Net shift"); plt.ylabel("number of images")
plt.title("Per-image context shift"); plt.tight_layout(); plt.show()

## 12. Try it on any image

Classifies a picture among **all** classes (base + new) with the hand-written prompt and with the learned
prompt, side by side. Change `URL` to any image link. The image should show something from the class list.

In [ ]:
URL = "http://images.cocodataset.org/val2017/000000000285.jpg"   # a bear (a CIFAR-100 class)

r = requests.get(URL, timeout=30); r.raise_for_status()
img = Image.open(BytesIO(r.content)).convert("RGB")
with torch.no_grad():
    f = F.normalize(clip_model.encode_image(preprocess(img).unsqueeze(0).to(device)), dim=-1)   # [1, 512]
    probs_h = (logit_scale * f @ handcrafted_text_features(classnames).T).softmax(-1)[0]   # hand-written
    probs_l = cocoop_logits(f, classnames).softmax(-1)[0]          # learned, image-conditional prompt

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].imshow(img); ax[0].axis("off")
for a, p, title in [(ax[1], probs_h, "hand-written prompt"), (ax[2], probs_l, "CoCoOp prompt")]:
    top = p.topk(5)                                       # 5 most likely classes
    a.barh([classnames[i] for i in top.indices.tolist()][::-1], top.values.cpu().numpy()[::-1])
    a.set_xlim(0, 1); a.set_title(title)
plt.tight_layout(); plt.show()

## 13. Save the learned prompt

`state_dict()` collects all trainable tensors. The file is tiny. To reuse it later: rebuild
`PromptLearner` with the same settings and call `prompt_learner.load_state_dict(torch.load(path))`.
Colab deletes `/content` when the session ends, so download the file or copy it to Drive.

In [ ]:
path = f"/content/cocoop_prompt_{DATASET}_{SHOTS}shot.pt"    # file name includes the settings
torch.save({"state_dict": prompt_learner.state_dict(), "backbone": BACKBONE,
            "ctx_init": CTX_INIT, "n_ctx": prompt_learner.n_ctx, "results": results}, path)
print("Saved", path, f"({os.path.getsize(path)/1024:.0f} KB)")

## Things to try

- **Batch size 1** (paper setting): slower but closer to the paper's results.
- **Zero-init the Meta-Net's last layer** (add `nn.init.zeros_(self.meta_net.linear2.weight); nn.init.zeros_(self.meta_net.linear2.bias)`
  at the end of `__init__`): training then starts exactly at zero-shot quality. Not in the paper; see if it helps.
- **ViT-B/16** backbone in both notebooks, to compare with the paper's numbers.
- **Fewer shots** (4 or 8): does the gap between CoOp and CoCoOp on new classes grow or shrink?
- **Your own dataset** (`DATASET = "folder"`), e.g. defect types: train on some defect classes and test on
  unseen ones. This is the base-to-new setting applied to your domain.
- **Next methods in this line of work:** MaPLe (prompts in both encoders), KgCoOp and ProGrad (keep learned
  prompts close to the hand-written one), PromptSRC (self-regularisation).